### Name : Moataz Mahmoud Soliman Ibrahim
#### DataSet: My Company internal dataset related to steel production
##### Descreption:



---
# Step 1: Data Loading, Inspection & Cleaning
---

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

## Phase 1: Data Loading and Initial Inspection

### 1.1 Load Primary Heat Data

In [3]:
df = pd.read_csv('../data/raw/flat_raw_data.csv')
print(f"Shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
print(df.columns.tolist())
print(f"\nData Types:")
print(df.dtypes)
df.head()

Shape: (23348, 39)

Columns (39):
['RC', 'HID', 'STOP_DATE', 'STEEL_GRADE', 'TAPPED_WGT', 'TOTAL_CHARGE', 'DRI_CHARGE', 'SCRAP_CHARGE', 'POWER_ON_TIME', 'FINAL_TEMP', 'TOTAL_C_DRI', 'LOCAL_SCRAP', 'HOME_RETURN_SCRAP', 'IMPORTED_SCRAP', 'PIG_IRON', 'DRI_BRIQUETTED', 'FESI', 'FEMN', 'SIMN', 'FECR', 'NICKEL', 'FETI', 'FEB', 'FENB', 'FEP', 'FEMO', 'CU', 'FEV', 'NITROVAN', 'ELEC_ENERGY', 'OXYGEN', 'COKE_LUMP', 'MODULE_CARBON', 'COKE_POWDER', 'BURNT_LIME', 'BURNT_DOLAMITE', 'FLUX', 'C_ANALYSIS', ' ']

Data Types:
RC                     int64
HID                    int64
STOP_DATE                str
STEEL_GRADE           object
TAPPED_WGT           float64
TOTAL_CHARGE         float64
DRI_CHARGE           float64
SCRAP_CHARGE         float64
POWER_ON_TIME        float64
FINAL_TEMP           float64
TOTAL_C_DRI          float64
LOCAL_SCRAP          float64
HOME_RETURN_SCRAP    float64
IMPORTED_SCRAP       float64
PIG_IRON             float64
DRI_BRIQUETTED       float64
FESI                   

,RC,HID,STOP_DATE,STEEL_GRADE,TAPPED_WGT,TOTAL_CHARGE,DRI_CHARGE,SCRAP_CHARGE,POWER_ON_TIME,FINAL_TEMP,TOTAL_C_DRI,LOCAL_SCRAP,HOME_RETURN_SCRAP,IMPORTED_SCRAP,PIG_IRON,DRI_BRIQUETTED,FESI,FEMN,SIMN,FECR,NICKEL,FETI,FEB,FENB,FEP,FEMO,CU,FEV,NITROVAN,ELEC_ENERGY,OXYGEN,COKE_LUMP,MODULE_CARBON,COKE_POWDER,BURNT_LIME,BURNT_DOLAMITE,FLUX,C_ANALYSIS,
0,268683,533422,01-JAN-2023 00.26.48,2067,162.40,186.50,150.50,36.00,45.30,1669.00,NaN,22.00,14.00,NaN,NaN,NaN,0,1001,0,0,0,0,0,0,0,0,0,0,0,92700,6788,978,3675,0,1996,7000,0,NaN,
1,268687,533423,01-JAN-2023 01.20.28,2067,161.40,189.60,150.60,39.00,41.00,1659.00,NaN,NaN,39.00,NaN,NaN,NaN,0,999,0,0,0,0,0,0,0,0,0,0,0,88900,6326,567,3652,0,1996,7000,0,NaN,
2,268690,533424,01-JAN-2023 02.12.57,1015,159.90,190.50,144.50,46.00,42.40,1647.00,NaN,NaN,46.00,NaN,NaN,NaN,0,800,0,0,0,0,0,0,0,0,0,0,0,88000,6075,432,3552,0,1497,7000,0,NaN,
3,268693,533425,01-JAN-2023 03.06.08,1015,159.40,191.20,152.20,39.00,40.80,1642.00,NaN,NaN,39.00,NaN,NaN,NaN,0,799,0,0,0,0,0,0,0,0,0,0,0,87000,6087,253,3411,0,1996,7000,0,NaN,
4,268696,533426,01-JAN-2023 03.53.08,1010,159.20,183.40,147.40,36.00,38.30,1630.00,NaN,22.00,14.00,NaN,NaN,NaN,0,255,0,0,0,0,0,0,0,0,0,0,0,83900,5861,237,3131,0,998,7000,0,NaN,


### 1.2 Load Steel Grade Mapping

In [4]:
grades_df = pd.read_csv('../data/raw/flat_steel_families.csv')
print(f"Shape: {grades_df.shape}")
print(f"\nColumns: {grades_df.columns.tolist()}")
grades_df.head(10)

Shape: (128, 2)

Columns: ['GRADE CODE', 'Steel Family ']


,GRADE CODE,Steel Family
0,4300,M-C
1,7855,M-C
2,4200,M-C
3,7010,M-C
4,6074,M-C
5,6075,M-C
6,1070,M-C
7,9994,M-C
8,1071,M-C
9,1072,M-C


In [6]:
# Standardize grade mapping column names
grades_df.columns = grades_df.columns.str.strip()
grades_df = grades_df.rename(columns={'GRADE CODE': 'STEEL_GRADE', 'Steel Family': 'STEEL_FAMILY'})
# If column names differ (changed by the plant system), try variations. The file will be always generated from the plant with exactly two columns, so we can infer the correct names based on content if needed.
if 'STEEL_GRADE' not in grades_df.columns:
    grades_df.columns = ['STEEL_GRADE', 'STEEL_FAMILY']
grades_df['STEEL_GRADE'] = grades_df['STEEL_GRADE'].astype(str).str.strip()
grades_df['STEEL_FAMILY'] = grades_df['STEEL_FAMILY'].astype(str).str.strip()

print(f"Steel families distribution:")
print(grades_df['STEEL_FAMILY'].value_counts())

# Verify alignment with primary dataset
# The mapping master in comming from quality so sometimes we need to get the difference to include it or to drop it later in the cleaning step.
primary_grades = set(df['STEEL_GRADE'].astype(str).str.strip().unique())
mapping_grades = set(grades_df['STEEL_GRADE'].unique())
unmatched = primary_grades - mapping_grades
print(f"\nGrades in primary but not in mapping: {len(unmatched)}")
if unmatched:
    print(f"Unmatched grades: {sorted(unmatched)}")

Steel families distribution:
STEEL_FAMILY
L-C    96
M-C    32
Name: count, dtype: int64

Grades in primary but not in mapping: 9
Unmatched grades: ['5041', '6081', '8888', '9903', '9991', '9992', '9993', '9996 NEW', '9999']


In [8]:
# Get count of records that has unmatched grades (these records are the steel production records that will be missing the steel family)
unmatched_count = df[df['STEEL_GRADE'].astype(str).str.strip().isin(unmatched)].shape[0]
print(f"Number of records with unmatched grades: {unmatched_count}")

Number of records with unmatched grades: 53


### 1.3 Load DRI Analysis Data

The DRI analysis contains daily chemical composition of Direct Reduced Iron, including:
- **C (Carbon %)**: Carbon content in DRI 
- **SIO2 (Silica %)**: Silica content, indicator of DRI quality (lower is better)
- **TFE, MET, MFE**: Iron content and metallization metrics

**Matching Logic**: Each heat's STOP_DATE is matched to the DRI analysis from **2 days prior**,
because there is a ~2 day lag between DRI production analysis and its use in the EAF.

In [9]:
dr_df = pd.read_csv('../data/raw/dr_analysis.csv')
print(f"DRI Analysis Shape: {dr_df.shape}")
print(f"\nColumns: {dr_df.columns.tolist()}")
print(f"\nData Types:")
print(dr_df.dtypes)
print(f"\nKey columns preview:")
dr_df[['ANALYSIS_DATE', 'C', 'SIO2', 'TFE', 'MET', 'MFE']].head(10)

DRI Analysis Shape: (1144, 21)

Columns: ['ANALYSIS_DATE', 'TFE', 'SIO2', 'CAO', 'MGO', 'MNO', 'AL2O3', 'P2O5', 'V2O5', 'S', 'K2O', 'TIO2', 'CR2O3', 'NA2O', 'CAOSIO', 'MFE', 'MET', 'C', 'FEO', 'FE2O3', 'FE3O4']

Data Types:
ANALYSIS_DATE        str
TFE              float64
SIO2             float64
CAO              float64
MGO              float64
MNO              float64
AL2O3            float64
P2O5             float64
V2O5             float64
S                float64
K2O              float64
TIO2             float64
CR2O3            float64
NA2O             float64
CAOSIO           float64
MFE              float64
MET              float64
C                float64
FEO              float64
FE2O3            float64
FE3O4            float64
dtype: object

Key columns preview:


,ANALYSIS_DATE,C,SIO2,TFE,MET,MFE
0,29-12-2022,2.50,2.10,90.75,93.89,85.20
1,30-12-2022,2.39,2.01,90.82,94.29,85.64
2,31-12-2022,2.51,2.05,90.38,92.66,83.74
3,01-01-2023,2.57,1.94,90.52,95.00,86.00
4,02-01-2023,2.58,1.95,91.26,95.47,87.13
5,03-01-2023,2.57,1.99,91.26,95.69,87.35
6,04-01-2023,2.62,1.31,91.63,96.50,88.44
7,05-01-2023,2.65,0.00,91.28,94.51,86.28
8,06-01-2023,2.51,1.98,90.78,94.98,86.22
9,07-01-2023,2.42,2.05,90.34,94.04,84.98


In [10]:
# Parse DRI analysis dates
dr_df['ANALYSIS_DATE'] = pd.to_datetime(dr_df['ANALYSIS_DATE'], dayfirst=True)
print(f"DRI Analysis date range: {dr_df['ANALYSIS_DATE'].min()} to {dr_df['ANALYSIS_DATE'].max()}")
print(f"Total DRI analysis records: {len(dr_df)}")
print(f"\nDRI Carbon (C) statistics:")
print(dr_df['C'].describe())
print(f"\nDRI Silica (SIO2) statistics:")
print(dr_df['SIO2'].describe())

DRI Analysis date range: 2022-12-29 00:00:00 to 2026-04-05 00:00:00
Total DRI analysis records: 1144

DRI Carbon (C) statistics:
count   1144.00
mean       2.32
std        0.19
min        1.08
25%        2.21
50%        2.33
75%        2.44
max        2.79
Name: C, dtype: float64

DRI Silica (SIO2) statistics:
count   1144.00
mean       1.65
std        0.50
min        0.00
25%        1.59
50%        1.73
75%        1.91
max        2.50
Name: SIO2, dtype: float64


In [11]:
# Check for zero-value rows in DRI analysis (some dates have zeros for all composition except TFE/MFE/MET/C)
zero_sio2_mask = dr_df['SIO2'] == 0
print(f"DRI analysis rows with SIO2 = 0: {zero_sio2_mask.sum()} out of {len(dr_df)}")
print(f"DRI analysis rows with valid composition: {(~zero_sio2_mask).sum()}")
print(f"\nNote: Rows with SIO2=0 still have valid C (carbon) values.")
print(f"C values where SIO2=0: mean={dr_df.loc[zero_sio2_mask, 'C'].mean():.3f}")
print(f"C values where SIO2>0: mean={dr_df.loc[~zero_sio2_mask, 'C'].mean():.3f}")

DRI analysis rows with SIO2 = 0: 69 out of 1144
DRI analysis rows with valid composition: 1075

Note: Rows with SIO2=0 still have valid C (carbon) values.
C values where SIO2=0: mean=2.274
C values where SIO2>0: mean=2.318


### 1.4 Match DRI Analysis to Heats (STOP_DATE - 2 Days)

For each heat, we look up the DRI analysis from 2 days before the heat's stop date (production end date for the heat).
Example: if a heat stopped on Jan 4, 2023, we use the DRI analysis from Jan 2, 2023.

In [12]:
# Parse STOP_DATE in heat data temporarily for matching
df['STOP_DATE_PARSED'] = pd.to_datetime(df['STOP_DATE'], format='%d-%b-%Y %H.%M.%S')

# Calculate the lookup date: STOP_DATE - 2 days (date only, no time)
df['DRI_LOOKUP_DATE'] = df['STOP_DATE_PARSED'].dt.normalize() - pd.Timedelta(days=2)

# Normalize DRI analysis date for matching
dr_df['ANALYSIS_DATE_NORM'] = dr_df['ANALYSIS_DATE'].dt.normalize()

# Select key DRI columns to merge
dri_merge_cols = ['ANALYSIS_DATE_NORM', 'C', 'SIO2', 'TFE', 'MET', 'MFE']
dr_merge = dr_df[dri_merge_cols].copy()
dr_merge = dr_merge.rename(columns={
    'ANALYSIS_DATE_NORM': 'DRI_LOOKUP_DATE',
    'C': 'DRI_CARBON_PCT',
    'SIO2': 'DRI_SIO2_PCT',
    'TFE': 'DRI_TFE',
    'MET': 'DRI_MET',
    'MFE': 'DRI_MFE'
})

# Merge
rows_before_merge = len(df)
df = df.merge(dr_merge, on='DRI_LOOKUP_DATE', how='left')

# Report matching results
matched = df['DRI_CARBON_PCT'].notna().sum()
unmatched_dri = df['DRI_CARBON_PCT'].isna().sum()
print(f"DRI Analysis Matching Results:")
print(f"  Total heats: {len(df)}")
print(f"  Matched with DRI analysis: {matched} ({matched/len(df)*100:.1f}%)")
print(f"  Unmatched (no DRI analysis for lookup date): {unmatched_dri} ({unmatched_dri/len(df)*100:.1f}%)")
print(f"\nDRI Carbon in matched heats:")
print(df['DRI_CARBON_PCT'].describe())
print(f"\nDRI Silica in matched heats:")
print(df['DRI_SIO2_PCT'].describe())

DRI Analysis Matching Results:
  Total heats: 23348
  Matched with DRI analysis: 22522 (96.5%)
  Unmatched (no DRI analysis for lookup date): 826 (3.5%)

DRI Carbon in matched heats:
count   22522.00
mean        2.32
std         0.19
min         1.40
25%         2.22
50%         2.34
75%         2.45
max         2.79
Name: DRI_CARBON_PCT, dtype: float64

DRI Silica in matched heats:
count   22522.00
mean        1.64
std         0.51
min         0.00
25%         1.58
50%         1.72
75%         1.90
max         2.50
Name: DRI_SIO2_PCT, dtype: float64


In [13]:
# Show some examples of the date matching
examples = df[['HID', 'STOP_DATE', 'DRI_LOOKUP_DATE', 'DRI_CARBON_PCT', 'DRI_SIO2_PCT']].head(10)
print("Date matching examples (STOP_DATE → LOOKUP_DATE = STOP_DATE - 2 days):")
examples

Date matching examples (STOP_DATE → LOOKUP_DATE = STOP_DATE - 2 days):


,HID,STOP_DATE,DRI_LOOKUP_DATE,DRI_CARBON_PCT,DRI_SIO2_PCT
0,533422,01-JAN-2023 00.26.48,2022-12-30,2.39,2.01
1,533423,01-JAN-2023 01.20.28,2022-12-30,2.39,2.01
2,533424,01-JAN-2023 02.12.57,2022-12-30,2.39,2.01
3,533425,01-JAN-2023 03.06.08,2022-12-30,2.39,2.01
4,533426,01-JAN-2023 03.53.08,2022-12-30,2.39,2.01
5,533427,01-JAN-2023 04.41.18,2022-12-30,2.39,2.01
6,533428,01-JAN-2023 05.31.20,2022-12-30,2.39,2.01
7,533429,01-JAN-2023 06.17.47,2022-12-30,2.39,2.01
8,533430,01-JAN-2023 07.29.50,2022-12-30,2.39,2.01
9,533431,01-JAN-2023 08.35.10,2022-12-30,2.39,2.01


In [14]:
# Clean up temporary columns
df = df.drop(columns=['STOP_DATE_PARSED', 'DRI_LOOKUP_DATE'])

### 1.5 Initial Data Profiling

In [15]:
# Summary statistics for numeric columns
df.describe()

,RC,HID,TAPPED_WGT,TOTAL_CHARGE,DRI_CHARGE,SCRAP_CHARGE,POWER_ON_TIME,FINAL_TEMP,TOTAL_C_DRI,LOCAL_SCRAP,HOME_RETURN_SCRAP,IMPORTED_SCRAP,PIG_IRON,DRI_BRIQUETTED,FESI,FEMN,SIMN,FECR,NICKEL,FETI,FEB,FENB,FEP,FEMO,CU,FEV,NITROVAN,ELEC_ENERGY,OXYGEN,COKE_LUMP,MODULE_CARBON,COKE_POWDER,BURNT_LIME,BURNT_DOLAMITE,FLUX,C_ANALYSIS,DRI_CARBON_PCT,DRI_SIO2_PCT,DRI_TFE,DRI_MET,DRI_MFE
count,23348.00,23348.00,23348.00,22797.00,23348.00,22797.00,23348.00,22399.00,0.00,18508.00,20423.00,2232.00,1.00,1.00,23348.00,23348.00,23348.00,23348.00,23348.00,23348.00,23348.00,23348.00,23348.00,23348.00,23348.00,23348.00,23348.00,23348.00,23348.00,23348.00,23348.00,23348.00,23348.00,23348.00,23348.00,81.00,22522.00,22522.00,22522.00,22522.00,22522.00
mean,304853.11,545380.94,53890.74,183.91,146.99,36.70,41.82,1646.52,NaN,23.13,17.58,19.39,40.00,40.00,0.01,694.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,87296.95,5588.66,117.99,2839.24,0.00,1282.85,5324.28,1.91,0.04,2.32,1.64,91.25,94.58,86.32
std,20633.46,7593.73,76135.89,27.30,28.02,6.21,4.68,25.63,NaN,7.08,9.13,10.50,NaN,NaN,0.84,380.42,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,7517.69,556.20,288.90,687.17,0.00,847.60,1026.07,16.89,0.01,0.19,0.51,0.59,0.94,1.20
min,268683.00,456687.00,13.60,31.34,0.00,3.00,0.00,1451.00,NaN,0.09,0.96,0.56,40.00,40.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.02,1.40,0.00,89.18,90.72,81.33
25%,287295.25,539566.75,160.40,180.80,143.80,33.99,39.50,1633.00,NaN,20.90,13.61,11.41,40.00,40.00,0.00,303.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,83900.00,5284.00,0.00,2352.00,0.00,499.00,5000.00,0.00,0.04,2.22,1.58,90.85,93.98,85.59
50%,305008.50,545398.50,163.40,186.00,150.10,36.00,41.40,1649.00,NaN,22.00,14.00,18.43,40.00,40.00,0.00,806.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,87000.00,5586.00,0.00,2795.50,0.00,1500.00,5500.00,0.00,0.04,2.34,1.72,91.30,94.64,86.35
75%,322691.50,551213.25,160000.00,190.58,155.30,38.00,43.60,1663.00,NaN,23.83,15.75,25.61,40.00,40.00,0.00,999.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,90500.00,5897.00,0.00,3266.00,0.00,1999.00,6000.00,0.00,0.05,2.45,1.90,91.70,95.21,87.13
max,340314.00,745801.00,194000.00,3146.99,3108.00,150.00,170.70,1755.00,NaN,77.00,150.00,78.00,40.00,40.00,128.00,5866.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,178900.00,10683.00,4227.00,10691.00,0.00,9415.00,23600.00,420.00,0.07,2.79,2.50,92.87,97.30,89.73


In [16]:
# Count Records of tapped wgt less than 140 and bigger than 180, these are basically plant constraints, that should not happen in normal operation.
tapped_wgt_low = df[df['TOTAL_CHARGE'] < 140].shape[0]
tapped_wgt_high = df[df['TOTAL_CHARGE'] > 210].shape[0]
print(f"Number of records with  < 140: {tapped_wgt_low}")
print(f"Number of records with  > 180: {tapped_wgt_high}")

Number of records with  < 140: 526
Number of records with  > 180: 364


In [17]:
# Unique values for categorical columns
print(f"Unique STEEL_GRADE values: {df['STEEL_GRADE'].nunique()}")
print(f"Unique HID values: {df['HID'].nunique()}")

# Date range
dates = pd.to_datetime(df['STOP_DATE'], format='%d-%b-%Y %H.%M.%S')
print(f"\nDate range: {dates.min()} to {dates.max()}")

# Duplicate Heat IDs
dup_count = df['HID'].duplicated().sum()
print(f"\nDuplicate HID count: {dup_count}")

Unique STEEL_GRADE values: 144
Unique HID values: 23297

Date range: 2023-01-01 00:26:48 to 2025-12-31 23:27:24

Duplicate HID count: 51


---
## Phase 2: Data Cleaning

### 2.1 Column Name and Type Standardization

In [19]:
# Strip whitespace, uppercase, replace spaces with underscores
df.columns = df.columns.str.strip().str.upper().str.replace(' ', '_')

# Define expected columns from the project specification including DRI analysis columns, there are other columns that is not useful for the analysis, so we will drop them later.
expected_columns = [
    'HID', 'STOP_DATE', 'STEEL_GRADE', 'TOTAL_CHARGE', 'DRI_CHARGE',
    'SCRAP_CHARGE', 'LOCAL_SCRAP', 'HOME_RETURN_SCRAP', 'IMPORTED_SCRAP',
    'ELEC_ENERGY', 'OXYGEN', 'COKE_LUMP', 'MODULE_CARBON', 'COKE_POWDER',
    'BURNT_LIME', 'BURNT_DOLAMITE', 'POWER_ON_TIME', 'FINAL_TEMP', 'TAPPED_WGT',
    # DRI analysis columns
    'DRI_CARBON_PCT', 'DRI_SIO2_PCT', 'DRI_TFE', 'DRI_MET', 'DRI_MFE'
]

# Identify columns to drop (not in expected list)
cols_to_drop = [c for c in df.columns if c not in expected_columns]
print(f"Columns to drop ({len(cols_to_drop)}): {cols_to_drop}")
df = df.drop(columns=cols_to_drop)

print(f"\nRemaining columns ({len(df.columns)}): {df.columns.tolist()}")

Columns to drop (0): []

Remaining columns (24): ['HID', 'STOP_DATE', 'STEEL_GRADE', 'TAPPED_WGT', 'TOTAL_CHARGE', 'DRI_CHARGE', 'SCRAP_CHARGE', 'POWER_ON_TIME', 'FINAL_TEMP', 'LOCAL_SCRAP', 'HOME_RETURN_SCRAP', 'IMPORTED_SCRAP', 'ELEC_ENERGY', 'OXYGEN', 'COKE_LUMP', 'MODULE_CARBON', 'COKE_POWDER', 'BURNT_LIME', 'BURNT_DOLAMITE', 'DRI_CARBON_PCT', 'DRI_SIO2_PCT', 'DRI_TFE', 'DRI_MET', 'DRI_MFE']


In [ ]:
# Convert types
df['STOP_DATE'] = pd.to_datetime(df['STOP_DATE'], format='%d-%b-%Y %H.%M.%S')
df['STEEL_GRADE'] = df['STEEL_GRADE'].astype(str).str.strip()
df['HID'] = df['HID'].astype(str).str.strip()

# Convert measurement columns to numeric (coerce errors to NaN)
numeric_cols = [
    'TOTAL_CHARGE', 'DRI_CHARGE', 'SCRAP_CHARGE', 'LOCAL_SCRAP',
    'HOME_RETURN_SCRAP', 'IMPORTED_SCRAP', 'ELEC_ENERGY', 'OXYGEN',
    'COKE_LUMP', 'MODULE_CARBON', 'COKE_POWDER', 'BURNT_LIME',
    'BURNT_DOLAMITE', 'POWER_ON_TIME', 'FINAL_TEMP', 'TAPPED_WGT',
    'DRI_CARBON_PCT', 'DRI_SIO2_PCT', 'DRI_TFE', 'DRI_MET', 'DRI_MFE'
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("Data types after conversion:")
print(df.dtypes)

Data types after conversion:
HID                          object
STOP_DATE            datetime64[ns]
STEEL_GRADE                  object
TAPPED_WGT                  float64
TOTAL_CHARGE                float64
DRI_CHARGE                  float64
SCRAP_CHARGE                float64
POWER_ON_TIME               float64
FINAL_TEMP                  float64
LOCAL_SCRAP                 float64
HOME_RETURN_SCRAP           float64
IMPORTED_SCRAP              float64
ELEC_ENERGY                   int64
OXYGEN                        int64
COKE_LUMP                     int64
MODULE_CARBON                 int64
COKE_POWDER                   int64
BURNT_LIME                    int64
BURNT_DOLAMITE                int64
DRI_CARBON_PCT              float64
DRI_SIO2_PCT                float64
DRI_TFE                     float64
DRI_MET                     float64
DRI_MFE                     float64
dtype: object


In [ ]:
# Fix mixed units in TAPPED_WGT: some values are in kg instead of tons
# Values > 1000 are clearly in kg (a valid EAF tap is ~100-200 tons); divide those by 1000
kg_mask = df['TAPPED_WGT'] > 1000
kg_count = kg_mask.sum()
df.loc[kg_mask, 'TAPPED_WGT'] = df.loc[kg_mask, 'TAPPED_WGT'] / 1000

print(f"TAPPED_WGT unit fix: converted {kg_count} rows from kg -> tons")
print(df['TAPPED_WGT'].describe())

TAPPED_WGT unit fix: converted 7774 rows from kg -> tons
count   23348.00
mean      161.08
std         6.69
min        13.60
25%       159.50
50%       161.70
75%       163.70
max       194.00
Name: TAPPED_WGT, dtype: float64


### 2.2 Missing Value Analysis and Treatment

In [ ]:
# Missing value analysis
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print(f"Columns with missing values: {len(missing)}")
missing

Columns with missing values: 11


,missing_count,missing_pct
IMPORTED_SCRAP,21116,90.44
LOCAL_SCRAP,4840,20.73
HOME_RETURN_SCRAP,2925,12.53
FINAL_TEMP,949,4.06
DRI_TFE,826,3.54
DRI_SIO2_PCT,826,3.54
DRI_CARBON_PCT,826,3.54
DRI_MET,826,3.54
DRI_MFE,826,3.54
SCRAP_CHARGE,551,2.36


In [ ]:
rows_before = len(df)

# 1. Drop rows missing critical columns
critical_cols = ['HID', 'STOP_DATE', 'TOTAL_CHARGE', 'ELEC_ENERGY', 'TAPPED_WGT']
df = df.dropna(subset=critical_cols)
print(f"Dropped {rows_before - len(df)} rows with missing critical values")

# 2. Fill additive columns with 0
additive_cols = [
    'COKE_LUMP', 'MODULE_CARBON', 'COKE_POWDER', 'BURNT_LIME',
    'BURNT_DOLAMITE', 'LOCAL_SCRAP', 'HOME_RETURN_SCRAP', 'IMPORTED_SCRAP'
]
for col in additive_cols:
    filled = df[col].isnull().sum()
    df[col] = df[col].fillna(0)
    if filled > 0:
        print(f"Filled {filled} NaN values in {col} with 0")

# 3. Other numeric columns - evaluate case by case
# Note: DRI analysis columns (DRI_CARBON_PCT, DRI_SIO2_PCT, etc.) are NOT filled here
# They will be handled separately since missing means no DRI analysis available.
other_numeric = [c for c in numeric_cols if c not in critical_cols and c not in additive_cols
                 and not c.startswith('DRI_') or c == 'DRI_CHARGE']
for col in other_numeric:
    if col not in df.columns:
        continue
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        pct = n_missing / len(df) * 100
        if pct < 5:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            print(f"Filled {n_missing} NaN in {col} with median ({median_val:.2f})")
        else:
            print(f"WARNING: {col} has {n_missing} missing ({pct:.1f}%) - dropping rows")
            df = df.dropna(subset=[col])

# 4. Handle DRI analysis missing values
dri_missing = df['DRI_CARBON_PCT'].isna().sum()
print(f"\nDRI analysis columns - missing after cleaning: {dri_missing} ({dri_missing/len(df)*100:.1f}%)")
print(f"Note: DRI analysis NaN rows will be dropped since DRI carbon is critical for this analysis.")

# Drop rows without DRI analysis match
df = df.dropna(subset=['DRI_CARBON_PCT'])
print(f"After dropping rows without DRI analysis: {len(df)} rows remain")

# Handle SIO2=0 rows: these have valid carbon but no detailed composition
# Keep them but flag
df['DRI_HAS_FULL_ANALYSIS'] = (df['DRI_SIO2_PCT'] > 0).astype(int)
print(f"Rows with full DRI analysis (SIO2 > 0): {df['DRI_HAS_FULL_ANALYSIS'].sum()}")
print(f"Rows with partial DRI analysis (SIO2 = 0): {(~df['DRI_HAS_FULL_ANALYSIS'].astype(bool)).sum()}")

print(f"\nFinal shape after missing value treatment: {df.shape}")
print(f"Remaining missing values: {df.isnull().sum().sum()}")

Dropped 551 rows with missing critical values
Filled 4289 NaN values in LOCAL_SCRAP with 0
Filled 2374 NaN values in HOME_RETURN_SCRAP with 0
Filled 20565 NaN values in IMPORTED_SCRAP with 0
Filled 915 NaN in FINAL_TEMP with median (1649.00)

DRI analysis columns - missing after cleaning: 821 (3.6%)
Note: DRI analysis NaN rows will be dropped since DRI carbon is critical for this analysis.
After dropping rows without DRI analysis: 21976 rows remain
Rows with full DRI analysis (SIO2 > 0): 20581
Rows with partial DRI analysis (SIO2 = 0): 1395

Final shape after missing value treatment: (21976, 25)
Remaining missing values: 0


### 2.3 Data Validation and Consistency Checks

In [ ]:
validation_report = {}

# 1. Charge Balance Check
charge_diff = abs(df['TOTAL_CHARGE'] - (df['DRI_CHARGE'] + df['SCRAP_CHARGE']))
tolerance = 1.0  # 1 ton tolerance
charge_imbalance = charge_diff > tolerance
validation_report['charge_balance_issues'] = charge_imbalance.sum()
print(f"1. Charge Balance: {charge_imbalance.sum()} heats with DRI+SCRAP != TOTAL_CHARGE (>{tolerance}t tolerance)")

# 2. Scrap Components Check  
scrap_sum = df['LOCAL_SCRAP'] + df['HOME_RETURN_SCRAP'] + df['IMPORTED_SCRAP']
scrap_diff = abs(df['SCRAP_CHARGE'] - scrap_sum)
scrap_imbalance = scrap_diff > tolerance
validation_report['scrap_component_issues'] = scrap_imbalance.sum()
print(f"2. Scrap Components: {scrap_imbalance.sum()} heats with component sum != SCRAP_CHARGE")

# 3. Yield Feasibility Check
yield_violation = df['TAPPED_WGT'] > df['TOTAL_CHARGE']
validation_report['yield_violations'] = yield_violation.sum()
print(f"3. Yield Feasibility: {yield_violation.sum()} heats where TAPPED_WGT > TOTAL_CHARGE")

# 4. Physical Range Checks
range_checks = {
    'ELEC_ENERGY': (40000, 150000),
    'FINAL_TEMP': (1500, 1750),
    'POWER_ON_TIME': (20, 120),
    'TOTAL_CHARGE': (50, 200)
}
print("\n4. Physical Range Checks:")
for col, (low, high) in range_checks.items():
    out_of_range = ((df[col] < low) | (df[col] > high)).sum()
    validation_report[f'{col}_out_of_range'] = out_of_range
    print(f"   {col} [{low}-{high}]: {out_of_range} heats out of range")

# 5. DRI Analysis Range Checks
print("\n5. DRI Analysis Range Checks:")
dri_range_checks = {
    'DRI_CARBON_PCT': (0.5, 5.0),
    'DRI_SIO2_PCT': (0, 5.0),
    'DRI_MET': (80, 100),
}
for col, (low, high) in dri_range_checks.items():
    valid = df[col].dropna()
    out_of_range = ((valid < low) | (valid > high)).sum()
    validation_report[f'{col}_out_of_range'] = out_of_range
    print(f"   {col} [{low}-{high}]: {out_of_range} values out of range")

# 6. Non-negativity Check
print("\n6. Non-Negativity Check:")
neg_count = 0
for col in numeric_cols:
    if col not in df.columns:
        continue
    negatives = (df[col] < 0).sum()
    if negatives > 0:
        print(f"   {col}: {negatives} negative values")
        neg_count += negatives
validation_report['negative_values'] = neg_count
if neg_count == 0:
    print("   No negative values found.")

1. Charge Balance: 0 heats with DRI+SCRAP != TOTAL_CHARGE (>1.0t tolerance)
2. Scrap Components: 172 heats with component sum != SCRAP_CHARGE
3. Yield Feasibility: 812 heats where TAPPED_WGT > TOTAL_CHARGE

4. Physical Range Checks:
   ELEC_ENERGY [40000-150000]: 32 heats out of range
   FINAL_TEMP [1500-1750]: 18 heats out of range
   POWER_ON_TIME [20-120]: 27 heats out of range
   TOTAL_CHARGE [50-200]: 1428 heats out of range

5. DRI Analysis Range Checks:
   DRI_CARBON_PCT [0.5-5.0]: 0 values out of range
   DRI_SIO2_PCT [0-5.0]: 0 values out of range
   DRI_MET [80-100]: 0 values out of range

6. Non-Negativity Check:
   No negative values found.


In [ ]:
# Flag validation issues (but don't remove - keep for investigation)
# Drop any old flag columns to avoid stale state on re-run
df = df.drop(columns=[c for c in df.columns if c.startswith('FLAG_')], errors='ignore')

df['FLAG_CHARGE_BALANCE'] = charge_imbalance.reindex(df.index, fill_value=False)
df['FLAG_YIELD_VIOLATION'] = yield_violation.reindex(df.index, fill_value=False)

for col, (low, high) in range_checks.items():
    df[f'FLAG_{col}_RANGE'] = (df[col] < low) | (df[col] > high)

flag_cols = [c for c in df.columns if c.startswith('FLAG_')]
df['FLAG_ANY_VALIDATION'] = df[flag_cols].any(axis=1)
print(f"Total heats with any validation flag: {df['FLAG_ANY_VALIDATION'].sum()} ({df['FLAG_ANY_VALIDATION'].mean()*100:.1f}%)")

Total heats with any validation flag: 2179 (9.9%)


### 2.4 Outlier Detection and Flagging (IQR Method)

In [ ]:
outlier_cols = ['ELEC_ENERGY', 'TOTAL_CHARGE', 'TAPPED_WGT', 'OXYGEN', 'POWER_ON_TIME', 'FINAL_TEMP']

outlier_stats = []
for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    is_outlier = (df[col] < lower) | (df[col] > upper)
    df[f'OUTLIER_{col}'] = is_outlier
    outlier_stats.append({
        'Column': col, 'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
        'Lower_Bound': lower, 'Upper_Bound': upper,
        'Outlier_Count': is_outlier.sum(),
        'Outlier_Pct': f"{is_outlier.mean()*100:.2f}%"
    })

# Overall outlier flag
outlier_flag_cols = [c for c in df.columns if c.startswith('OUTLIER_')]
df['OUTLIER_ANY'] = df[outlier_flag_cols].any(axis=1)

outlier_summary = pd.DataFrame(outlier_stats)
print(f"Overall: {df['OUTLIER_ANY'].sum()} heats ({df['OUTLIER_ANY'].mean()*100:.1f}%) flagged as outlier in at least one variable")
outlier_summary

Overall: 3303 heats (15.0%) flagged as outlier in at least one variable


,Column,Q1,Q3,IQR,Lower_Bound,Upper_Bound,Outlier_Count,Outlier_Pct
0,ELEC_ENERGY,83800.00,90400.00,6600.00,73900.00,100300.00,965,4.39%
1,TOTAL_CHARGE,180.89,190.58,9.69,166.35,205.12,1699,7.73%
2,TAPPED_WGT,159.50,163.70,4.20,153.20,170.00,484,2.20%
3,OXYGEN,5284.00,5898.00,614.00,4363.00,6819.00,511,2.33%
4,POWER_ON_TIME,39.50,43.50,4.00,33.50,49.50,762,3.47%
5,FINAL_TEMP,1634.00,1663.00,29.00,1590.50,1706.50,610,2.78%


In [ ]:
# Drop any row that contains outliers in critical variables (ELEC_ENERGY, TOTAL_CHARGE, TAPPED_WGT)
critical_outlier_cols = ['OUTLIER_ELEC_ENERGY', 'OUTLIER_TOTAL_CHARGE', 'OUTLIER_TAPPED_WGT']

rows_before_drop = len(df)
critical_outlier_mask = df[critical_outlier_cols].any(axis=1)

print("Outliers in critical variables (before drop):")
for col in critical_outlier_cols:
    print(f"  {col}: {df[col].sum()}")

df = df[~critical_outlier_mask].copy()
print(f"\nDropped {rows_before_drop - len(df)} rows with outliers in critical variables")
print(f"Remaining rows: {len(df)}")

Outliers in critical variables (before drop):
  OUTLIER_ELEC_ENERGY: 965
  OUTLIER_TOTAL_CHARGE: 1699
  OUTLIER_TAPPED_WGT: 484

Dropped 2459 rows with outliers in critical variables
Remaining rows: 19517


### 2.5 Merge Steel Grade Information

In [ ]:
# Merge steel family (drop any pre-existing variants to avoid _x/_y conflicts on re-run)
df = df.drop(columns=[c for c in df.columns if c.startswith('STEEL_FAMILY')], errors='ignore')
df = df.merge(grades_df[['STEEL_GRADE', 'STEEL_FAMILY']], on='STEEL_GRADE', how='left')

# Standardize steel family names
family_map = {
    'H-C': 'HIGH_CARBON',
    'M-C': 'MEDIUM_CARBON',
    'L-C': 'LOW_CARBON'
}
df['STEEL_FAMILY'] = df['STEEL_FAMILY'].map(family_map).fillna('UNKNOWN')

print("Steel Family Distribution:")
print(df['STEEL_FAMILY'].value_counts())

unmatched_count = (df['STEEL_FAMILY'] == 'UNKNOWN').sum()
print(f"\nUnmatched grades (UNKNOWN): {unmatched_count}")
if unmatched_count > 0:
    unmatched_grades = df.loc[df['STEEL_FAMILY'] == 'UNKNOWN', 'STEEL_GRADE'].unique()
    print(f"Unmatched grade codes: {sorted(unmatched_grades)}")

Steel Family Distribution:
STEEL_FAMILY
LOW_CARBON       19279
MEDIUM_CARBON      195
UNKNOWN             43
Name: count, dtype: int64

Unmatched grades (UNKNOWN): 43
Unmatched grade codes: ['5041', '6081', '8888', '9903', '9991', '9992', '9993', '9996 NEW', '9999']


### Save Cleaned Dataset

In [ ]:
# Save cleaned data (includes DRI analysis columns)
flag_and_outlier_cols = [c for c in df.columns if c.startswith('FLAG_') or c.startswith('OUTLIER_')]

import os
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/cleaned_heat_data_with_dri_carbon.csv', index=False)
print(f"Saved cleaned dataset: {df.shape}")
print(f"  Includes {len(flag_and_outlier_cols)} flag/outlier columns for reference")
print(f"  Includes DRI analysis columns: DRI_CARBON_PCT, DRI_SIO2_PCT, DRI_TFE, DRI_MET, DRI_MFE")
print(f"\nFinal columns: {df.columns.tolist()}")

Saved cleaned dataset: (19517, 40)
  Includes 14 flag/outlier columns for reference
  Includes DRI analysis columns: DRI_CARBON_PCT, DRI_SIO2_PCT, DRI_TFE, DRI_MET, DRI_MFE

Final columns: ['HID', 'STOP_DATE', 'STEEL_GRADE', 'TAPPED_WGT', 'TOTAL_CHARGE', 'DRI_CHARGE', 'SCRAP_CHARGE', 'POWER_ON_TIME', 'FINAL_TEMP', 'LOCAL_SCRAP', 'HOME_RETURN_SCRAP', 'IMPORTED_SCRAP', 'ELEC_ENERGY', 'OXYGEN', 'COKE_LUMP', 'MODULE_CARBON', 'COKE_POWDER', 'BURNT_LIME', 'BURNT_DOLAMITE', 'DRI_CARBON_PCT', 'DRI_SIO2_PCT', 'DRI_TFE', 'DRI_MET', 'DRI_MFE', 'DRI_HAS_FULL_ANALYSIS', 'FLAG_CHARGE_BALANCE', 'FLAG_YIELD_VIOLATION', 'FLAG_ELEC_ENERGY_RANGE', 'FLAG_FINAL_TEMP_RANGE', 'FLAG_POWER_ON_TIME_RANGE', 'FLAG_TOTAL_CHARGE_RANGE', 'FLAG_ANY_VALIDATION', 'OUTLIER_ELEC_ENERGY', 'OUTLIER_TOTAL_CHARGE', 'OUTLIER_TAPPED_WGT', 'OUTLIER_OXYGEN', 'OUTLIER_POWER_ON_TIME', 'OUTLIER_FINAL_TEMP', 'OUTLIER_ANY', 'STEEL_FAMILY']


In [ ]:
# Showing descriptive statistics for the cleaned dataset
df.describe()

,STOP_DATE,TAPPED_WGT,TOTAL_CHARGE,DRI_CHARGE,SCRAP_CHARGE,POWER_ON_TIME,FINAL_TEMP,LOCAL_SCRAP,HOME_RETURN_SCRAP,IMPORTED_SCRAP,ELEC_ENERGY,OXYGEN,COKE_LUMP,MODULE_CARBON,COKE_POWDER,BURNT_LIME,BURNT_DOLAMITE,DRI_CARBON_PCT,DRI_SIO2_PCT,DRI_TFE,DRI_MET,DRI_MFE,DRI_HAS_FULL_ANALYSIS
count,19517,19517.00,19517.00,19517.00,19517.00,19517.00,19517.00,19517.00,19517.00,19517.00,19517.00,19517.00,19517.00,19517.00,19517.00,19517.00,19517.00,19517.00,19517.00,19517.00,19517.00,19517.00,19517.00
mean,2024-07-08 09:45:48.682225664,161.59,186.17,149.80,36.37,41.53,1646.36,18.48,15.61,1.99,87124.31,5600.14,93.72,2804.44,0.00,1297.58,5285.68,2.33,1.64,91.26,94.60,86.34,0.94
min,2023-01-01 00:26:48,153.30,166.39,100.00,18.00,21.40,1451.00,0.00,0.00,0.00,74000.00,2920.00,0.00,805.00,0.00,0.00,0.00,1.40,0.00,89.18,90.72,81.33,0.00
25%,2023-10-09 15:40:59,159.60,181.59,145.10,33.99,39.50,1633.00,17.72,12.83,0.00,84000.00,5305.00,0.00,2339.00,0.00,500.00,5000.00,2.22,1.58,90.85,94.01,85.61,1.00
50%,2024-08-04 23:46:57,161.80,186.10,150.30,36.00,41.30,1649.00,22.00,14.00,0.00,87000.00,5593.00,0.00,2773.00,0.00,1500.00,5500.00,2.34,1.72,91.31,94.64,86.37,1.00
75%,2025-03-29 14:55:48,163.70,190.39,155.20,37.99,43.20,1662.00,22.73,15.55,0.00,90000.00,5886.00,0.00,3223.00,0.00,1999.00,5700.00,2.45,1.90,91.70,95.22,87.17,1.00
max,2025-12-31 22:23:30,169.90,205.10,181.70,82.98,66.50,1755.00,77.00,82.00,69.00,100300.00,10683.00,3343.00,7018.00,0.00,6791.00,23600.00,2.79,2.50,92.87,97.30,89.73,1.00
std,NaN,2.99,6.95,8.75,5.67,2.84,24.60,10.99,10.01,6.77,4644.70,458.87,226.09,633.57,0.00,837.35,955.39,0.18,0.51,0.59,0.92,1.19,0.24


In [ ]:
# Data Quality Report Summary
print("=" * 60)
print("DATA QUALITY REPORT (WITH DRI CARBON ANALYSIS)")
print("=" * 60)
print(f"Original rows: {rows_before}")
print(f"Final rows: {len(df)}")
print(f"Rows removed: {rows_before - len(df)}")
print(f"\nDRI Analysis Integration:")
print(f"  DRI Carbon (C%) range: {df['DRI_CARBON_PCT'].min():.3f} - {df['DRI_CARBON_PCT'].max():.3f}")
print(f"  DRI Silica (SIO2%) range: {df['DRI_SIO2_PCT'].min():.3f} - {df['DRI_SIO2_PCT'].max():.3f}")
print(f"  Rows with full DRI analysis: {df['DRI_HAS_FULL_ANALYSIS'].sum()}")
print(f"\nValidation Issues:")
for k, v in validation_report.items():
    print(f"  {k}: {v}")
print(f"\nOutlier Summary:")
print(f"  Heats with any outlier: {df['OUTLIER_ANY'].sum()} ({df['OUTLIER_ANY'].mean()*100:.1f}%)")
print(f"\nSteel Family Distribution:")
for fam, count in df['STEEL_FAMILY'].value_counts().items():
    print(f"  {fam}: {count} ({count/len(df)*100:.1f}%)")

DATA QUALITY REPORT (WITH DRI CARBON ANALYSIS)
Original rows: 23348
Final rows: 19517
Rows removed: 3831

DRI Analysis Integration:
  DRI Carbon (C%) range: 1.395 - 2.790
  DRI Silica (SIO2%) range: 0.000 - 2.497
  Rows with full DRI analysis: 18312

Validation Issues:
  charge_balance_issues: 0
  scrap_component_issues: 172
  yield_violations: 812
  ELEC_ENERGY_out_of_range: 32
  FINAL_TEMP_out_of_range: 18
  POWER_ON_TIME_out_of_range: 27
  TOTAL_CHARGE_out_of_range: 1428
  DRI_CARBON_PCT_out_of_range: 0
  DRI_SIO2_PCT_out_of_range: 0
  DRI_MET_out_of_range: 0
  negative_values: 0

Outlier Summary:
  Heats with any outlier: 844 (4.3%)

Steel Family Distribution:
  LOW_CARBON: 19279 (98.8%)
  MEDIUM_CARBON: 195 (1.0%)
  UNKNOWN: 43 (0.2%)


: 